# 🎵 Treinamento e Preparação de Dados Musical com Persistência Contínua
### Projeto Integrador VI — Sistemas de Informação (PUC-Campinas, 2026) — Grupo B6

Este notebook foi configurado para **salvar cada arquivo em tempo real diretamente no seu Google Drive** (limite otimizado de **20.000 músicas** para execução rápida e segura sem risco de perda de dados).

---
## 1. Verificação da GPU T4

In [ ]:
!nvidia-smi

Fri Nov  6 18:02:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 2. Instalação das Dependências

In [ ]:
!pip install -q torch transformers pretty_midi mido music21 pyyaml tqdm pandas numpy scikit-learn matplotlib

---
## 3. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 4. Descompactar o `lmd_full.tar.gz` para o SSD do Colab

In [ ]:
import os

!mkdir -p /content/SI-PI6-2026-G06/arquivos_brutos
!mkdir -p /content/SI-PI6-2026-G06/configuracao
!mkdir -p /content/SI-PI6-2026-G06/dados_processados
!mkdir -p /content/SI-PI6-2026-G06/modelos
!mkdir -p /content/SI-PI6-2026-G06/scripts

caminho_tar = '/content/drive/MyDrive/Colab Notebooks/SI-PI6-2026-G06/lmd_full.tar.gz'

# Se ainda não estiver descompactado, descompacta:
if not os.path.exists('/content/SI-PI6-2026-G06/arquivos_brutos/lmd_full'):
    print('⏳ Descompactando arquivos brutos (1 a 2 min)...')
    !tar -xzf "{caminho_tar}" -C /content/SI-PI6-2026-G06/arquivos_brutos/
    print('✅ Descompactação concluída!')
else:
    print('✅ Arquivos brutos já disponíveis no SSD do Colab!')

⏳ Descompactando arquivos brutos (1 a 2 min)...
✅ Descompactação concluída!


---
## 5. Configurar Parâmetros (Limite Seguro de 100.000 Arquivos)

In [ ]:
%%writefile /content/SI-PI6-2026-G06/configuracao/pipeline.yaml
dataset:
  pasta_lmd_full: "arquivos_brutos/lmd_full"
  arquivo_md5_json: "arquivos_brutos/md5_to_paths.json"
  extensoes: [".mid", ".midi"]

piloto:
  ativo: true
  quantidade: 100000     # Limite otimizado de 100000 músicas (~15 a 20 minutos de processamento)

filtros:
  notas_minimas: 32
  notas_maximas: 2000
  duracao_minima_seg: 15.0
  duracao_maxima_seg: 600.0
  polifonia_maxima_pct: 0.30
  pitch_minimo: 36              # C2
  pitch_maximo: 96              # C7

normalizacao:
  quantizar_duracao: true
  grades_ritmicas: [0.125, 0.25, 0.5, 1.0, 2.0]
  regra_monofonia: "pitch_mais_agudo"

divisao:
  treino: 0.80
  validacao: 0.10
  teste: 0.10
  semente: 42

saidas:
  pasta_midi_validos: "dados_processados/midi_validos"
  pasta_melodias: "dados_processados/melodias_extraidas"
  pasta_sequencias: "dados_processados/sequencias"
  pasta_treino: "dados_processados/treino"
  pasta_validacao: "dados_processados/validacao"
  pasta_teste: "dados_processados/teste"
  pasta_relatorios: "relatorios"

Overwriting /content/SI-PI6-2026-G06/configuracao/pipeline.yaml


---
## 6. Script de Preparação com **Persistência Imediata em Disco e Google Drive**
Cada arquivo processado é salvo **imediatamente** no disco e copiado para o Google Drive. Mesmo se você clicar em Stop a qualquer momento, nada é perdido!

In [ ]:
%%writefile /content/SI-PI6-2026-G06/scripts/pipeline_direto.py
import os, json, random, shutil
from pathlib import Path
import pretty_midi, yaml
from tqdm import tqdm

RAIZ = Path('/content/SI-PI6-2026-G06')
with open(RAIZ / 'configuracao' / 'pipeline.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

pasta_lmd = RAIZ / cfg['dataset']['pasta_lmd_full']
pasta_treino = RAIZ / cfg['saidas']['pasta_treino']
pasta_val = RAIZ / cfg['saidas']['pasta_validacao']
pasta_teste = RAIZ / cfg['saidas']['pasta_teste']
pasta_seq = RAIZ / cfg['saidas']['pasta_sequencias']

for p in [pasta_treino, pasta_val, pasta_teste, pasta_seq]:
    p.mkdir(parents=True, exist_ok=True)

# Pasta de Backup Permanente no Drive
drive_backup = Path('/content/drive/MyDrive/Colab Notebooks/SI-PI6-2026-G06/dados_processados')
for sub in ['treino', 'validacao', 'teste', 'sequencias']:
    (drive_backup / sub).mkdir(parents=True, exist_ok=True)

midis = list(pasta_lmd.rglob('*.mid')) + list(pasta_lmd.rglob('*.midi'))
print(f'Total de MIDIs brutos localizados: {len(midis):,}')

limite = cfg['piloto']['quantidade'] if cfg['piloto']['ativo'] else len(midis)
midis = midis[:limite]
print(f'🚀 Processando lote seguro de {len(midis):,} arquivos com persistência contínua...')

grades = cfg['normalizacao']['grades_ritmicas']
def quantizar(dur_beats):
    mais_proximo = min(grades, key=lambda g: abs(g - dur_beats))
    return f'{mais_proximo:.2f}'

vocab_set = set()
caminho_vocab_local = pasta_seq / 'vocabulario.json'
caminho_vocab_drive = drive_backup / 'sequencias' / 'vocabulario.json'

# Se já existir vocabulário anterior, carrega:
if caminho_vocab_local.exists():
    try:
        d = json.loads(caminho_vocab_local.read_text(encoding='utf-8'))
        vocab_set.update(d.get('token_para_id', {}).keys())
    except Exception:
        pass

def salvar_vocabulario():
    lista_vocab = sorted([t for t in vocab_set if not t.startswith('<')]) + ['<PAD>', '<START>', '<END>']
    token_to_id = {tok: i for i, tok in enumerate(lista_vocab)}
    id_to_token = {str(i): tok for i, tok in enumerate(lista_vocab)}
    vocab_json = {'token_para_id': token_to_id, 'id_para_token': id_to_token, 'tamanho_vocabulario': len(lista_vocab)}
    caminho_vocab_local.write_text(json.dumps(vocab_json, indent=2), encoding='utf-8')
    try:
        caminho_vocab_drive.write_text(json.dumps(vocab_json, indent=2), encoding='utf-8')
    except Exception:
        pass

salvos_treino = 0
salvos_val = 0
salvos_teste = 0
random.seed(42)

try:
    for idx, mid_path in enumerate(tqdm(midis, desc='Filtrando e Salvando em Tempo Real')):
        try:
            pm = pretty_midi.PrettyMIDI(str(mid_path))
            if pm.get_end_time() < cfg['filtros']['duracao_minima_seg']:
                continue
            insts = [inst for inst in pm.instruments if not inst.is_drum and len(inst.notes) >= cfg['filtros']['notas_minimas']]
            if not insts:
                continue
            best_inst = max(insts, key=lambda x: len(x.notes))
            tempos = pm.get_tempo_changes()
            bpm = float(tempos[1][0]) if len(tempos[1]) > 0 else 120.0
            sec_per_beat = 60.0 / bpm

            notas = sorted(best_inst.notes, key=lambda n: n.start)
            tokens = []
            ultimo_fim = 0.0

            for n in notas:
                if not (cfg['filtros']['pitch_minimo'] <= n.pitch <= cfg['filtros']['pitch_maximo']):
                    continue
                if n.start > ultimo_fim + 0.05:
                    pausa_beats = (n.start - ultimo_fim) / sec_per_beat
                    if pausa_beats >= 0.12:
                        t_rest = f'REST_{quantizar(pausa_beats)}'
                        tokens.append(t_rest)
                        vocab_set.add(t_rest)
                dur_beats = (n.end - n.start) / sec_per_beat
                t_note = f'NOTE_{n.pitch}'
                t_dur = f'DURATION_{quantizar(dur_beats)}'
                tokens.extend([t_note, t_dur])
                vocab_set.add(t_note)
                vocab_set.add(t_dur)
                ultimo_fim = max(ultimo_fim, n.end)

            if len(tokens) >= 32:
                dado = {'md5': mid_path.stem, 'bpm': bpm, 'tokens': tokens}
                r = random.random()
                if r < 0.80:
                    subpasta = 'treino'
                    salvos_treino += 1
                elif r < 0.90:
                    subpasta = 'validacao'
                    salvos_val += 1
                else:
                    subpasta = 'teste'
                    salvos_teste += 1

                nome_arquivo = f'{mid_path.stem}.json'
                arq_local = RAIZ / 'dados_processados' / subpasta / nome_arquivo
                arq_local.write_text(json.dumps(dado), encoding='utf-8')

                # Salvar vocabulário a cada 50 músicas
                if (salvos_treino + salvos_val + salvos_teste) % 50 == 0:
                    salvar_vocabulario()
        except Exception:
            pass
except KeyboardInterrupt:
    print('\n⚠️ Processamento pausado pelo usuário. Salvando dados processados até agora...')

# Gravação final garantida do vocabulário e sincronização com Drive
salvar_vocabulario()
print(f'\n✅ Persistência Concluída!')
print(f'   Treino     : {salvos_treino:,} músicas salvas')
print(f'   Validação  : {salvos_val:,} músicas salvas')
print(f'   Teste      : {salvos_teste:,} músicas salvas')
print(f'   Vocabulário: {len(vocab_set)} tokens únicos salvos em vocabulario.json')

Overwriting /content/SI-PI6-2026-G06/scripts/pipeline_direto.py


---
## 7. Executar a Preparação dos Dados (Salva Tudo em Tempo Real)

In [ ]:
%cd /content/SI-PI6-2026-G06
!python scripts/pipeline_direto.py

/content/SI-PI6-2026-G06
Total de MIDIs brutos localizados: 178,561
🚀 Processando lote seguro de 20,000 arquivos com persistência contínua...
Filtrando e Salvando em Tempo Real:   0% 17/20000 [00:01<21:11, 15.71it/s]/usr/local/lib/python3.13/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
Filtrando e Salvando em Tempo Real: 100% 20000/20000 [23:19<00:00, 14.29it/s]

✅ Persistência Concluída!
   Treino     : 14,912 músicas salvas
   Validação  : 1,867 músicas salvas
   Teste      : 1,867 músicas salvas
   Vocabulário: 74 tokens únicos salvos em vocabulario.json


---
## 8. Treinar o Transformer Autoregressivo com GPU T4 (Salva Checkpoints Direto no Drive)

In [ ]:
import json, math, os, time
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Treinando no dispositivo: {device}')

# Localizar vocabulário (no Colab ou no Drive)
candidatos_vocab = [
    Path('/content/SI-PI6-2026-G06/dados_processados/sequencias/vocabulario.json'),
    Path('/content/drive/MyDrive/Colab Notebooks/SI-PI6-2026-G06/dados_processados/sequencias/vocabulario.json'),
]
caminho_vocab = next((p for p in candidatos_vocab if p.exists()), None)
if caminho_vocab is None:
    raise FileNotFoundError('Execute a Célula 7 primeiro para gerar os dados e o vocabulário.')

vocab_data = json.loads(caminho_vocab.read_text(encoding='utf-8'))
token_para_id = vocab_data['token_para_id']
vocab_size = len(token_para_id)
print(f'Vocabulário carregado com {vocab_size} tokens!')

# Arquitetura Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

class MusicTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, nhead: int = 4, num_layers: int = 4, dim_feedforward: int = 512, dropout: float = 0.1, max_seq_len: int = 1024):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=x.device), diagonal=1)
        embeds = self.embedding(x) * math.sqrt(self.d_model)
        x_pos = self.pos_encoder(embeds)
        out = self.transformer(x_pos, mask=mask, is_causal=True)
        return self.fc_out(out)

class DatasetSequencias(Dataset):
    def __init__(self, pasta_dados: Path, seq_len: int = 128):
        self.seq_len = seq_len
        self.pad_id = token_para_id.get('<PAD>', 0)
        self.start_id = token_para_id.get('<START>', 0)
        self.end_id = token_para_id.get('<END>', 0)
        self.amostras = []
        for arq in sorted(pasta_dados.glob('*.json')):
            try:
                tokens = json.loads(arq.read_text(encoding='utf-8')).get('tokens', [])
                if len(tokens) >= 8:
                    ids = [self.start_id] + [token_para_id.get(t, self.pad_id) for t in tokens] + [self.end_id]
                    for i in range(0, max(1, len(ids) - seq_len), max(1, seq_len // 2)):
                        janela = ids[i : i + seq_len + 1]
                        if len(janela) > 4:
                            self.amostras.append(janela)
            except Exception:
                pass

    def __len__(self):
        return len(self.amostras)

    def __getitem__(self, idx):
        seq = self.amostras[idx]
        if len(seq) < self.seq_len + 1:
            seq = seq + [self.pad_id] * (self.seq_len + 1 - len(seq))
        else:
            seq = seq[: self.seq_len + 1]
        return torch.tensor(seq[:-1], dtype=torch.long), torch.tensor(seq[1:], dtype=torch.long)

pasta_treino = caminho_vocab.parent.parent / 'treino'
pasta_val = caminho_vocab.parent.parent / 'validacao'

ds_treino = DatasetSequencias(pasta_treino)
ds_val = DatasetSequencias(pasta_val)

if len(ds_treino) == 0:
    raise ValueError('Nenhuma amostra encontrada na pasta de treino. Execute a Célula 7.')

dl_treino = DataLoader(ds_treino, batch_size=32, shuffle=True, drop_last=True)
dl_val = DataLoader(ds_val, batch_size=32, shuffle=False)
print(f'Dataset pronto: Treino={len(ds_treino):,} amostras | Validação={len(ds_val):,} amostras')

modelo = MusicTransformer(vocab_size=vocab_size, d_model=256, nhead=4, num_layers=4, dim_feedforward=512).to(device)
criterio = nn.CrossEntropyLoss(ignore_index=token_para_id.get('<PAD>', 0))
otimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=15)

caminho_salvar_local = '/content/SI-PI6-2026-G06/modelos/modelo_transformer.pt'
caminho_salvar_drive = '/content/drive/MyDrive/Colab Notebooks/SI-PI6-2026-G06/modelos/modelo_transformer.pt'
os.makedirs(os.path.dirname(caminho_salvar_drive), exist_ok=True)

melhor_val_loss = float('inf')
epochs = 15

print('\n🚀 Iniciando treinamento do Transformer com GPU T4...')
for epoca in range(1, epochs + 1):
    t0 = time.time()
    modelo.train()
    total_loss = 0.0
    for x, y in dl_treino:
        x, y = x.to(device), y.to(device)
        otimizador.zero_grad()
        logits = modelo(x)
        loss = criterio(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        otimizador.step()
        total_loss += loss.item()

    scheduler.step()
    media_treino_loss = total_loss / len(dl_treino)

    modelo.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in dl_val:
            x, y = x.to(device), y.to(device)
            val_loss += criterio(modelo(x).view(-1, vocab_size), y.view(-1)).item()

    media_val_loss = val_loss / max(1, len(dl_val))
    print(f'Época {epoca:02d}/{epochs:02d} | Treino Loss: {media_treino_loss:.4f} | Val Loss: {media_val_loss:.4f} | Tempo: {time.time()-t0:.1f}s')

    if media_val_loss < melhor_val_loss:
        melhor_val_loss = media_val_loss
        ckpt = {
            'state_dict': modelo.state_dict(),
            'vocab_size': vocab_size,
            'd_model': 256,
            'nhead': 4,
            'num_layers': 4,
            'dim_feedforward': 512,
            'max_seq_len': 1024,
        }
        torch.save(ckpt, caminho_salvar_local)
        try:
            torch.save(ckpt, caminho_salvar_drive)
        except Exception:
            pass
        print(f'  ★ Melhor modelo salvo (Disco + Google Drive)!')

print('\n🎉 Treinamento do Transformer concluído com sucesso!')

Treinando no dispositivo: cuda
Vocabulário carregado com 74 tokens!
Dataset pronto: Treino=512,531 amostras | Validação=62,806 amostras

🚀 Iniciando treinamento do Transformer com GPU T4...
Época 01/15 | Treino Loss: 1.4179 | Val Loss: 1.2447 | Tempo: 547.0s
  ★ Melhor modelo salvo (Disco + Google Drive)!
Época 02/15 | Treino Loss: 1.2789 | Val Loss: 1.2020 | Tempo: 549.7s
  ★ Melhor modelo salvo (Disco + Google Drive)!
Época 03/15 | Treino Loss: 1.2438 | Val Loss: 1.1801 | Tempo: 548.9s
  ★ Melhor modelo salvo (Disco + Google Drive)!
Época 04/15 | Treino Loss: 1.2209 | Val Loss: 1.1668 | Tempo: 548.9s
  ★ Melhor modelo salvo (Disco + Google Drive)!
Época 05/15 | Treino Loss: 1.2031 | Val Loss: 1.1613 | Tempo: 548.0s
  ★ Melhor modelo salvo (Disco + Google Drive)!
Época 06/15 | Treino Loss: 1.1882 | Val Loss: 1.1539 | Tempo: 548.8s
  ★ Melhor modelo salvo (Disco + Google Drive)!
Época 07/15 | Treino Loss: 1.1748 | Val Loss: 1.1427 | Tempo: 548.0s
  ★ Melhor modelo salvo (Disco + Google

---
## 9. Baixar o Modelo Treinado para o seu Computador

In [ ]:
from google.colab import files

caminho = '/content/SI-PI6-2026-G06/modelos/modelo_transformer.pt'
if os.path.exists(caminho):
    files.download(caminho)
    print('✅ Download iniciado com sucesso!')
else:
    print('⚠️ Arquivo não encontrado. Verifique a Célula 8.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download iniciado com sucesso!
